# AI Safety Lab — run on Colab

Runs the **AI Safety Lab** — a single, self-contained teaching page, not the
operator dashboard — and gives you a public URL to open it.

The lab is a small React app (`lab/`) that simulates a factory floor and
walks the real detection pipeline over it — camera check, object detection,
PPE check, safety rules, confirmation, decision — using the product's actual
thresholds. It makes **no network call of any kind**: the simulation, the
scenarios, and the tutor all run client-side, reading the same engine that
used to sit behind a real backend.

That means this notebook needs **no GPU, no model downloads, and no API
key** — it just builds a static site and serves it. A minute or two, not the
three to five the full dashboard notebook takes.

**If you want the whole product**, the Lab also ships inside the operator
dashboard, under *Training → Lab* — same simulation, same source, alongside
the ten real monitoring capabilities. Use `colab_run.ipynb` for that. This
notebook is the lab on its own: faster to start, and useful when the
dashboard and its models are not what you are here for.


## 1. Get the code

The repository is private, so this needs a GitHub token.

Create a **fine-grained** token at
github.com → Settings → Developer settings → Personal access tokens →
Fine-grained tokens:

- **Repository access:** Only select repositories → this repo
- **Permissions:** Repository permissions → **Contents: Read-only**
- **Expiration:** whatever suits — 7 days is plenty for testing

That is the least a clone needs. It cannot push, cannot touch other repos, and
expires on its own.

The token is read with `getpass`, kept in this session's environment only, and
handed to git through a credential helper — so it is never written into the
notebook, never stored in `.git/config`, and disappears when the runtime does.


In [ ]:
import getpass, os, subprocess

REPO   = "rootstocktechai-debug/vikasgroup_visual_analytics_fullstack_beta"
BRANCH = "palak"

token = getpass.getpass("GitHub token (input hidden): ").strip()

import shutil
if os.path.exists("/content/app"):
    shutil.rmtree("/content/app")

r = subprocess.run(
    ["git","clone","--depth","1","--branch",BRANCH,
     f"https://{token}@github.com/{REPO}.git","/content/app"],
    capture_output=True, text=True)

print("cloned" if r.returncode == 0 else r.stderr[-800:])
del token

!ls /content/app/lab/src

## 2. Build the lab

Only `lab/` — `frontend/` and `backend/` (the real product) are not touched.


In [ ]:
%%bash
cd /content/app/lab
node --version
npm install --no-audit --no-fund --silent 2>&1 | tail -2
npm run build 2>&1 | tail -6
ls -la dist/index.html

## 3. Serve it

No backend, so this is a plain static file server — the build output copied
under a `lab/` folder (matching the base path it was built with) and served
from the folder above it, so `/lab/assets/...` resolves the same way it does
in production.

Logs go to `/content/server.log`.


In [ ]:
import subprocess, time, urllib.request, shutil, pathlib

site_dir = pathlib.Path("/content/site")
if site_dir.exists():
    shutil.rmtree(site_dir)
site_dir.mkdir(parents=True)
shutil.copytree("/content/app/lab/dist", site_dir / "lab")

log = open("/content/server.log", "w")
server = subprocess.Popen(
    ["python3", "-m", "http.server", "8000"],
    cwd=str(site_dir), stdout=log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/lab/", timeout=2)
        print("lab is being served")
        break
    except Exception:
        time.sleep(1)
else:
    print("server failed to start — last of the log:")
    print(open("/content/server.log").read()[-2000:])

## 4. Open it

Two ways. **Try the first** — it is built into Colab, needs no account and no
external service. Both give you the lab's own `/lab/` path directly, not the
bare root, which has nothing behind it.


In [ ]:
# Option A — Colab's built-in port proxy (recommended)
# Works in the browser session you are signed into. Nothing to install.
from google.colab.output import eval_js
base = eval_js("google.colab.kernel.proxyPort(8000)")
print("OPEN THIS:", base.rstrip("/") + "/lab/")

**Option B — a public URL.** Use this instead if Option A misbehaves, or if you
want to open the lab on your phone or send it to someone. It publishes port
8000 on a temporary address that dies with this session.


In [ ]:
# Option B — cloudflared public URL (no account needed)
import re, subprocess, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared","tunnel","--url","http://localhost:8000","--no-autoupdate"],
    stdout=open("/content/tunnel.log","w"), stderr=subprocess.STDOUT)

url = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break

if url:
    print("\n" + "="*66)
    print("  OPEN THIS:", url.rstrip("/") + "/lab/")
    print("="*66)
else:
    print("Tunnel did not come up. Last lines of /content/tunnel.log:")
    print(open("/content/tunnel.log").read()[-1200:])
    print("\nUse Option A above instead.")

## 5. What to try

The whole lab is one page, Factory Floor A, watched by Camera 01. Everything
on it is live — drag a thing, flip a switch, and the pipeline on the right
re-runs in real time from the same rules the real product uses.

| Try | What it shows |
|---|---|
| Drag a worker into the red **Restricted Zone** | it takes 3 agreeing sightings before the box turns red — one bad frame is never enough |
| Click **Worker 03** (missing a helmet), then **Why?** | the pipeline's own numbered account of the verdict — scores, thresholds, votes |
| Click **Door 01** | it may stay open a few seconds before it is reported, then escalates the longer it stays open |
| Use the **Scenario** dropdown | one-click fault injection — a blocked walkway, a dark camera, an empty workstation — each with a restore button |
| Turn the lights down (the dark scenario) | the pipeline reports "cannot check," never a false "all clear" |
| **Add Worker** / **Move Forklift** / **Create Zone** | add to the floor yourself and watch the same rules apply to what you added |
| Ask the tutor a question | "why is Worker 03 flagged?", "what does the restricted zone do?", "why does it wait for 3 sightings?" — answered from the live frame, not a script |
| **Pause** / **0.5x** / **2x** | the simulation clock, independent of your browser's frame rate |


## 6. Editing while it runs

Pull new commits, rebuild, and redeploy the static files — the server itself
does not need restarting, since it reads from disk on every request.


In [ ]:
!cd /content/app && git pull --ff-only
!cd /content/app/lab && npm run build 2>&1 | tail -6

import shutil
shutil.rmtree("/content/site/lab")
shutil.copytree("/content/app/lab/dist", "/content/site/lab")
print("Rebuilt and redeployed. Refresh the lab tab.")

## 7. Known limits

- **This is a simulation, not the real models.** The engine uses the
  product's real thresholds and confirmation logic, but the "camera" is a
  drawn floor, not a YOLOv8/InsightFace pipeline over real video — that is
  the point of a teaching page, not a gap in it.
- **Nothing survives the session.** Colab wipes the machine when it
  disconnects, and the simulation itself resets on a page refresh — there is
  no history to lose, because none is kept.
- **This notebook does not run the operator dashboard or its backend.** See
  `deploy/colab_run.ipynb` for that, on a GPU, if you want the real product
  instead of the teaching page.


## 8. Stop everything


In [ ]:
for name in ("tunnel", "server"):
    proc = globals().get(name)
    if proc is None:
        print(name, "was not started")
        continue
    try:
        proc.terminate()
        print(name, "stopped")
    except Exception as e:
        print(name, "->", e)